# F1. Model catalog: facts without downloading

`facts()` reads `config.json` and the safetensors **headers** of a Hub repo over HTTP Range requests
(a few KB), never the weights. Results are cached for 7 days under `~/.cache/rightsize/models/`.

In [ ]:
from rightsize.catalog import facts

fx = facts("Qwen/Qwen3-4B")
fx.model_dump(exclude={"extra"})

Everything the KV-cache formula needs is here: layers, KV heads, head dim, max context.

In [ ]:
{
    k: fx.extra[k]
    for k in ("model_type", "params_by_dtype", "shards", "vocab_size", "tie_word_embeddings")
}

Gated repos work with `HF_TOKEN` in the environment. Offline mode serves the cache only:

In [ ]:
facts("Qwen/Qwen3-4B", offline=True).params_total

## How the header read works

A safetensors file starts with 8 bytes (little-endian length N) followed by N bytes of JSON that
lists every tensor's dtype and shape. Two Range requests are enough to count parameters.

In [ ]:
import httpx

from rightsize.catalog import safetensors_header

with httpx.Client(follow_redirects=True) as c:
    hdr = safetensors_header(
        c, "https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/model.safetensors"
    )
list(hdr.items())[:3]